In [1]:
import os, json, pickle, math, random
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
import torch.nn as nn

from functions import DoomDataset

In [2]:
CONTEXT = 512
STRIDE  = 256
MAX_TRANSPOSE = 6

BATCH_SIZE = 32
D_MODEL    = 256
N_HEADS    = 8
N_LAYERS   = 4
D_FF       = 1024
DROPOUT    = 0.15     # ważne przy małych danych
LR         = 3e-4
EPOCHS     = 100

PROC_DIR = '../data/processed/'

os.makedirs('../models', exist_ok=True)
os.makedirs('../output', exist_ok=True)

In [3]:
random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

Device: NVIDIA GeForce RTX 4070


In [4]:
with open(os.path.join(PROC_DIR, 'sequences.pkl'), 'rb') as f:
    data = pickle.load(f)

with open(os.path.join(PROC_DIR, 'vocab.json')) as f:
    token2id = json.load(f)
id2token = {y: x for x,y in token2id.items()}

sequences = list(data.values())
names = list(data.keys())
VOCAB_SIZE = len(token2id)
PAD_ID = token2id['PAD']
print(f'Sequences: {len(sequences)} | Vocab: {VOCAB_SIZE} | Tokens: {sum(len(s) for s in sequences):,}')

Sequences: 43 | Vocab: 579 | Tokens: 384,327


In [5]:
id2pitch = {}
for tok, i in token2id.items():
    if tok.startswith('NOTE_ON_') or tok.startswith('NOTE_OFF_'):
        id2pitch[i] = int(tok.split('_')[-1])
noteon_ids = { int(tok.split('_')[-1]): i for tok, i in token2id.items() if tok.startswith('NOTE_ON') }
noteoff_ids = { int(tok.split('_')[-1]): i for tok, i in token2id.items() if tok.startswith('NOTE_OFF') }

SHIFT_MAPS = {}
for off in range(-MAX_TRANSPOSE, MAX_TRANSPOSE+1):
    m = np.arange(VOCAB_SIZE)
    for pitch, idx in noteon_ids.items():
        np_ = pitch + off
        if 0 <= np_ <= 127:
            m[idx] = noteon_ids[np_]
    for pitch, idx in noteoff_ids.items():
        np_ = pitch + off
        if 0 <= np_ <= 127:
            m[idx] = noteoff_ids[np_]
    SHIFT_MAPS[off] = m

In [6]:
class TokenEmbedding(nn.Module):
    def __init__(self, d_model, n_tokens):
        super().__init__()
        self.embedding_layer = torch.nn.Embedding(num_embeddings = n_tokens, embedding_dim = d_model)
    
    def forward(self, x):
        return self.embedding_layer(x)


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len):
        super().__init__()
        positional_encoding = np.zeros((max_len, d_model))
        for pos in range(max_len):
            for i in range(0, d_model, 2):
                positional_encoding[pos, i] = np.sin(pos / (10000 ** (2*i / d_model)))
                if i+1 < d_model:
                    positional_encoding[pos, i+1] = np.cos(pos / (10000 ** (2*i / d_model)))
        positional_encoding = torch.from_numpy(positional_encoding).float()
        self.register_buffer('pe', positional_encoding)
        
    def forward(self, x):
        return x + self.pe[:x.size(1), :]
        

class MaskedSelfAttention(nn.Module):
    def __init__(self, d_model, head_dim):
        super().__init__()
        self.head_dim = head_dim
        self.q = nn.Linear(d_model, head_dim)
        self.k = nn.Linear(d_model, head_dim)
        self.v = nn.Linear(d_model, head_dim)
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, x, casual_mask):
        q, k, v = self.q(x), self.k(x), self.v(x)
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        scores = scores.masked_fill(~casual_mask, -1e9)
        weights = self.softmax(scores)
        return torch.bmm(weights, v)


class MaskedMultiHeadedSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.head_dim = d_model//n_heads
        self.heads = nn.ModuleList([MaskedSelfAttention(d_model, self.head_dim) for _ in range(n_heads)])
        self.out = nn.Linear(d_model, d_model)

    def forward(self, x, casual_mask):
        outs = [h(x, casual_mask) for h in self.heads]
        return self.out(torch.cat(outs, dim=2))


class FeedForward(nn.Module):
    def __init__(self, d_model, ff_dim):
        super().__init__()
        self.l1 = nn.Linear(d_model, ff_dim)
        self.l2 = nn.Linear(ff_dim, d_model)

    def forward(self, x):
        return self.l2(torch.relu(self.l1(x)))


class DecoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, ff_dim, dropout):
        super().__init__()
        self.mha = MaskedMultiHeadedSelfAttention(d_model, n_heads)
        self.ff = FeedForward(d_model, ff_dim)
        self.dropout = nn.Dropout(dropout)
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)

    def forward(self, x, casual_mask):
        x = x + self.mha(self.ln1(x), casual_mask)
        ff = self.ff(self.ln2(x))
        if self.training:
            ff = self.dropout(ff)
        return x + ff


class DecoderStack(nn.Module):
    def __init__(self, d_model, n_layers, n_heads, ff_dim, dropout):
        super().__init__()
        self.layers = nn.ModuleList([DecoderLayer(d_model, n_heads, ff_dim, dropout) for _ in range(n_layers)])

    def forward(self, x, casual_mask):
        for layer in self.layers:
            x = layer(x, casual_mask)
        return x


class DoomTransformer(nn.Module):
    def __init__(self, n_tokens, max_len=512, d_model=256, n_layers=4, n_heads=4, ff_dim=1024, dropout=0.1):
        super().__init__()
        self.n_tokens = n_tokens
        self.max_len = max_len
        self.d_model = d_model
        self.n_layers = n_layers
        self.n_heads = n_heads
        self.ff_dim = ff_dim
        self.dropout = dropout

        self.token_embedding = TokenEmbedding(d_model, n_tokens)
        self.positional_encoding = PositionalEncoding(d_model, max_len)
        self.layer_normalization = nn.LayerNorm(d_model)
        self.decoder = DecoderStack(d_model, n_layers, n_heads, ff_dim, dropout)
        self.lm_head = nn.Linear(d_model, n_tokens)

    def forward(self, x):
        s = x.size(1)
        casual_mask = torch.tril(torch.ones(s, s, device=x.device, dtype=torch.bool))
        x = self.token_embedding(x)
        x = self.positional_encoding(x)
        x = self.layer_normalization(x)
        x = self.decoder(x, casual_mask)
        return self.lm_head(x)

    def save_checkpoint(self, path):
        print(f'Saving checkpoint {path}')
        torch.save({
            'number_of_tokens': self.n_tokens,
            'max_sequence_length': self.max_len,
            'embedding_dimension': self.d_model,
            'number_of_layers': self.n_layers,
            'number_of_heads': self.n_heads,
            'feed_forward_dimension': self.ff_dim,
            'dropout_rate': self.dropout,
            'model_state_dict': self.state_dict()
        }, path)

    @staticmethod
    def load_checkpoint(path) -> 'DoomTransformer':
        checkpoint = torch.load(path)
        model = DoomTransformer(
            checkpoint['number_of_tokens'],
            checkpoint['max_sequence_length'],
            checkpoint['embedding_dimension'], 
            checkpoint['number_of_layers'], 
            checkpoint['number_of_heads'],
            checkpoint['feed_forward_dimension'],
            checkpoint['dropout_rate']
        )
        model.load_state_dict(checkpoint['model_state_dict'])
        return model.to(device)